> 在线讲义：[第 4 章 · 配对比较：同一批初始条件交替跑，省下一半样本](https://wikieden.github.io/math-and-eval-stats/lessons/04-paired-comparison.html) · 本书《数学与实验统计》

# 第 4 章 · 配对比较：同一批初始条件交替跑，省下一半样本

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/04-paired-comparison.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 不配对 | 双比例 z | `unpaired_z(22, 30, 15, 30)` | z = 1.859，p = 0.063 |
| 第三步 精确 | Binomial(9, 0.5) | `mcnemar_exact(8, 1)` | 0.039 |
| 第三步 卡方 | 连续性校正 | `mcnemar_chi2(8, 1)` | 4.00，p = 0.046 |
| 第四步 配对 SE | 式 (7) | `paired_se(table_to_diffs(TABLE))` | 0.0920 |
| 第四步 模拟 | 难度 u | `simulate_difficulty()` | ρ = 0.270，方差比 0.730 |

例子：30 个初始条件，A、B 在同一批上各跑一次（编的四格表）：
都成功 14、A 成 B 败 8、A 败 B 成 1、都失败 7。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np
from scipy.stats import binomtest, chi2, norm
TABLE = {"both": 14, "a_only": 8, "b_only": 1, "neither": 7}

**函数 `unpaired_z`**：不配对：双比例 z 检验（第 2 章），返回 (z, 双侧 p)。

In [ ]:
def unpaired_z(k1: int, n1: int, k2: int, n2: int) -> tuple[float, float]:
    """不配对：双比例 z 检验（第 2 章），返回 (z, 双侧 p)。"""
    pbar = (k1 + k2) / (n1 + n2)
    se = math.sqrt(pbar * (1 - pbar) * (1 / n1 + 1 / n2))
    z = (k1 / n1 - k2 / n2) / se
    return z, float(2 * norm.sf(abs(z)))

**函数 `mcnemar_exact`**：McNemar 精确检验：只看不一致的 b + c 对，H₀ 下 b ~ Binomial(b + c, 0.5)。

In [ ]:
def mcnemar_exact(b: int, c: int) -> float:
    """McNemar 精确检验：只看不一致的 b + c 对，H₀ 下 b ~ Binomial(b + c, 0.5)。"""
    return float(binomtest(b, b + c, 0.5).pvalue)

**函数 `mcnemar_chi2`**：McNemar 卡方（连续性校正）：(|b − c| − 1)² / (b + c)，自由度 1。

In [ ]:
def mcnemar_chi2(b: int, c: int) -> tuple[float, float]:
    """McNemar 卡方（连续性校正）：(|b − c| − 1)² / (b + c)，自由度 1。"""
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    return stat, float(chi2.sf(stat, 1))

**函数 `paired_se`**：Miller 2024 式 (7)：配对差值的标准误 √(Var(d)/n)，Var 用 n − 1。

In [ ]:
def paired_se(diffs: np.ndarray) -> float:
    """Miller 2024 式 (7)：配对差值的标准误 √(Var(d)/n)，Var 用 n − 1。"""
    return float(np.sqrt(diffs.var(ddof=1) / len(diffs)))

**函数 `table_to_diffs`**：四格表 → 每个初始条件的差值 d = A − B（1、0、−1）。

In [ ]:
def table_to_diffs(t: dict) -> np.ndarray:
    """四格表 → 每个初始条件的差值 d = A − B（1、0、−1）。"""
    return np.array([0] * t["both"] + [1] * t["a_only"] + [-1] * t["b_only"] + [0] * t["neither"], dtype=float)

**函数 `variance_ratio`**：两组方差相同、相关系数 ρ 时：配对方差 / 不配对方差 = 1 − ρ。

In [ ]:
def variance_ratio(rho: float) -> float:
    """两组方差相同、相关系数 ρ 时：配对方差 / 不配对方差 = 1 − ρ。"""
    return 1 - rho

**函数 `simulate_difficulty`**：模拟"初始条件有难有易"：每个 IC 难度 u，A 成功率 0.95 − 0.9u，B 成功率 0.85 − 0.9u。返回 (A、B 结果的相关系数, 配对 / 不配对方差比)。

In [ ]:
def simulate_difficulty(n: int = 200000, seed: int = 0) -> tuple[float, float]:
    """模拟"初始条件有难有易"：每个 IC 难度 u，A 成功率 0.95 − 0.9u，B 成功率 0.85 − 0.9u。返回 (A、B 结果的相关系数, 配对 / 不配对方差比)。"""
    rng = np.random.default_rng(seed)
    u = rng.random(n)
    a = (rng.random(n) < 0.95 - 0.9 * u).astype(float)
    b = (rng.random(n) < 0.85 - 0.9 * u).astype(float)
    rho = float(np.corrcoef(a, b)[0, 1])
    return rho, float((a - b).var() / (a.var() + b.var()))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
t = TABLE
ka, kb, n = t["both"] + t["a_only"], t["both"] + t["b_only"], sum(t.values())

In [ ]:
print("\n第一步 同一批初始条件上的四格表")
print(f"  A 成功 {ka}/{n}，B 成功 {kb}/{n}；不一致的对：A 成 B 败 {t['a_only']}，A 败 B 成 {t['b_only']}")

In [ ]:
print("\n第二步 当成两组独立数据（不配对）")
z, p = unpaired_z(ka, n, kb, n)
print(f"  z = {z:.3f}，p = {p:.3f}")

In [ ]:
print("\n第三步 McNemar：只看不一致的那几对")
print(f"  精确检验 p = {mcnemar_exact(t['a_only'], t['b_only']):.3f}")
s, p2 = mcnemar_chi2(t["a_only"], t["b_only"])
print(f"  卡方（连续性校正）= {s:.2f}，p = {p2:.3f}")

In [ ]:
print("\n第四步 配对的标准误（Miller 式 7）")
d = table_to_diffs(t)
print(f"  差值均值 {d.mean():.3f}，配对 SE = {paired_se(d):.4f}")
se_unpaired = math.sqrt(ka / n * (1 - ka / n) / n + kb / n * (1 - kb / n) / n)
print(f"  不配对 SE = {se_unpaired:.4f}")

In [ ]:
print("\n第五步 配对为什么省：方差里减掉 2Cov")
for rho in (0.0, 0.3, 0.5):
    print(f"  相关 ρ = {rho}：配对方差是不配对的 {variance_ratio(rho):.0%}")
rho, ratio = simulate_difficulty()
print(f"  模拟（IC 有难有易）：A、B 结果相关 {rho:.3f}，方差比 {ratio:.3f}")

## 3. 代码里才看得到的

1. **配对 SE 用差值算**：不是分别算 A、B 的方差再相加，而是先每个初始条件做差，再算这 30 个差值的标准差。协方差自动被扣掉了。
2. **`ddof=1`**：样本方差除以 n − 1（Miller 式 (7) 也是 n − 1），n 小时和除以 n 差别不小。
3. **模拟里的 ρ 只有 0.27**：结果是 0 / 1，即使成功概率高度相关，单次结果的相关也会被抛硬币的随机性冲淡。真机上难度差异越大（有的 IC 几乎必败），ρ 越高。

## 4. 自己改一改

1. 把 `simulate_difficulty` 里的 0.9u 改成 0.3u（初始条件难度差不多），看 ρ 和方差比怎么变。
2. 把 `TABLE` 里的"都成功"从 14 改成 20、"都失败"从 7 改成 1，McNemar 的 p 值变不变？不配对的 p 值呢？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 同一张卷子比高低，初始条件对齐还要交替盲测；
> 当成独立来检验，难易波动白算进误差；
> McNemar 只看一对一错，H₀ 下一半对一半；
> 方差减掉两倍协方差，相关 ρ 越大省得越多。